# Analisis canasta SEPA
Se excluyen los meses:


In [1]:
# ============================================================
# CELDA 1 — Imports y configuración
# ============================================================
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Rutas en Colab
INPUT_DIR = "/content"
OUTPUT_FILE = "/content/canasta_SEPA_consolidado.xlsx"
IPC_FILE = "/content/IPC.xlsx"

# Archivos esperados
SEMESTRES = ["2022A","2022B","2023A","2023B","2024A","2024B","2025A","2025B","2026A"]

# Meses a descartar (cobertura insuficiente en SEPA)
MESES_DESCARTAR = ['2023-01','2023-02','2023-03','2023-04']

print("Archivos esperados:")
for s in SEMESTRES:
    path = f"{INPUT_DIR}/canasta_{s}_serie.xlsx"
    if os.path.exists(path):
        print(f"  ✅ canasta_{s}_serie.xlsx  ({os.path.getsize(path)/1024:.0f} KB)")
    else:
        print(f"  ❌ NO ENCONTRADO: canasta_{s}_serie.xlsx")

print(f"\nIPC INDEC:")
if os.path.exists(IPC_FILE):
    print(f"  ✅ IPC.xlsx  ({os.path.getsize(IPC_FILE)/1024:.0f} KB)")
else:
    print(f"  ❌ NO ENCONTRADO: IPC.xlsx")

print(f"\nMeses a descartar (cobertura SEPA insuficiente): {MESES_DESCARTAR}")

Archivos esperados:
  ✅ canasta_2022A_serie.xlsx  (126 KB)
  ✅ canasta_2022B_serie.xlsx  (130 KB)
  ✅ canasta_2023A_serie.xlsx  (157 KB)
  ✅ canasta_2023B_serie.xlsx  (217 KB)
  ✅ canasta_2024A_serie.xlsx  (239 KB)
  ✅ canasta_2024B_serie.xlsx  (229 KB)
  ✅ canasta_2025A_serie.xlsx  (225 KB)
  ✅ canasta_2025B_serie.xlsx  (229 KB)
  ✅ canasta_2026A_serie.xlsx  (154 KB)

IPC INDEC:
  ✅ IPC.xlsx  (24 KB)

Meses a descartar (cobertura SEPA insuficiente): ['2023-01', '2023-02', '2023-03', '2023-04']


In [2]:
# ============================================================
# CELDA 2 — Consolidar canasta nacional ponderada (todos los semestres)
# ============================================================
serie_nacional = []

for sem in SEMESTRES:
    path = f"{INPUT_DIR}/canasta_{sem}_serie.xlsx"
    if not os.path.exists(path):
        print(f"⚠️ Salteando {sem} (archivo no encontrado)")
        continue

    df = pd.read_excel(path, sheet_name='canasta_nacional_ponderada')
    df['semestre'] = sem
    serie_nacional.append(df)
    print(f"  {sem}: {len(df)} meses cargados")

serie_nacional = pd.concat(serie_nacional, ignore_index=True)
serie_nacional = serie_nacional.sort_values('mes').reset_index(drop=True)

# Marcar y filtrar meses descartados
serie_nacional['descartado'] = serie_nacional['mes'].isin(MESES_DESCARTAR)
serie_nacional_valida = serie_nacional[~serie_nacional['descartado']].copy()

# Recalcular variación mensual sobre la serie ya filtrada (continua)
serie_nacional_valida = serie_nacional_valida.sort_values('mes').reset_index(drop=True)
serie_nacional_valida['variacion_mensual_%'] = (
    serie_nacional_valida['canasta_nacional_ponderada'].pct_change() * 100
).round(2)

# Construir índice base 100 (primer mes válido = 100)
base = serie_nacional_valida['canasta_nacional_ponderada'].iloc[0]
serie_nacional_valida['indice_canasta_base100'] = (
    serie_nacional_valida['canasta_nacional_ponderada'] / base * 100
).round(2)

print(f"\n✅ Serie consolidada: {len(serie_nacional)} meses totales")
print(f"   Descartados: {serie_nacional['descartado'].sum()}")
print(f"   Válidos: {len(serie_nacional_valida)}")
print(f"\nPrimera fila válida (base 100): {serie_nacional_valida['mes'].iloc[0]}")
print(f"Última fila: {serie_nacional_valida['mes'].iloc[-1]}")

  2022A: 6 meses cargados
  2022B: 6 meses cargados
  2023A: 6 meses cargados
  2023B: 6 meses cargados
  2024A: 6 meses cargados
  2024B: 6 meses cargados
  2025A: 6 meses cargados
  2025B: 6 meses cargados
  2026A: 4 meses cargados

✅ Serie consolidada: 52 meses totales
   Descartados: 4
   Válidos: 48

Primera fila válida (base 100): 2022-01
Última fila: 2026-04


In [3]:
# ============================================================
# CELDA 3 — Consolidar canasta mensual por provincia
# ============================================================
serie_provincia = []

for sem in SEMESTRES:
    path = f"{INPUT_DIR}/canasta_{sem}_serie.xlsx"
    if not os.path.exists(path):
        continue

    df = pd.read_excel(path, sheet_name='canasta_mes_provincia')
    df['semestre'] = sem
    serie_provincia.append(df)

serie_provincia = pd.concat(serie_provincia, ignore_index=True)
serie_provincia['descartado'] = serie_provincia['mes'].isin(MESES_DESCARTAR)
serie_provincia_valida = serie_provincia[~serie_provincia['descartado']].copy()
serie_provincia_valida = serie_provincia_valida.sort_values(['mes','provincia']).reset_index(drop=True)

print(f"✅ Serie por provincia: {len(serie_provincia_valida):,} filas")
print(f"   Provincias: {serie_provincia_valida['provincia'].nunique()}")
print(f"   Meses: {serie_provincia_valida['mes'].nunique()}")

✅ Serie por provincia: 1,152 filas
   Provincias: 24
   Meses: 48


In [4]:
# ============================================================
# CELDA 4 — Consolidar canasta mensual por región
# ============================================================
serie_region = []

for sem in SEMESTRES:
    path = f"{INPUT_DIR}/canasta_{sem}_serie.xlsx"
    if not os.path.exists(path):
        continue

    df = pd.read_excel(path, sheet_name='canasta_mes_region')
    df['semestre'] = sem
    serie_region.append(df)

serie_region = pd.concat(serie_region, ignore_index=True)
serie_region['descartado'] = serie_region['mes'].isin(MESES_DESCARTAR)
serie_region_valida = serie_region[~serie_region['descartado']].copy()
serie_region_valida = serie_region_valida.sort_values(['mes','region']).reset_index(drop=True)

print(f"✅ Serie por región: {len(serie_region_valida):,} filas")
print(f"   Regiones: {serie_region_valida['region'].nunique()}")
print(f"   Meses: {serie_region_valida['mes'].nunique()}")

✅ Serie por región: 288 filas
   Regiones: 6
   Meses: 48


In [5]:
# ============================================================
# CELDA 5 — Cargar y procesar IPC INDEC
# ============================================================
ipc_raw = pd.read_excel(IPC_FILE)
print(f"IPC INDEC cargado: {len(ipc_raw)} filas")
print(f"Columnas: {list(ipc_raw.columns)}\n")

# Normalizar columna de fecha
# El formato es "ene-2017", "feb-2017", etc.
MES_MAP = {
    'ene':'01','feb':'02','mar':'03','abr':'04','may':'05','jun':'06',
    'jul':'07','ago':'08','sept':'09','sep':'09','oct':'10','nov':'11','dic':'12'
}

def parse_fecha_es(s):
    s = str(s).strip().lower()
    partes = s.split('-')
    if len(partes) != 2:
        return None
    mes_abr, anio = partes
    mes_abr = mes_abr.strip()
    if mes_abr not in MES_MAP:
        return None
    return f"{anio.strip()}-{MES_MAP[mes_abr]}"

ipc_raw['mes'] = ipc_raw['date'].apply(parse_fecha_es)
ipc_raw = ipc_raw.dropna(subset=['mes']).copy()

# Renombrar columnas para que sean más cómodas de usar
ipc_raw = ipc_raw.rename(columns={
    'Nivel general': 'ipc_general',
    'Alimentos y bebidas no alcohólicas': 'ipc_alimentos',
    'Bebidas Alcohólicas y tabaco': 'ipc_bebidas_alc',
    'Prendas de vestir y calzado': 'ipc_vestir',
    'Vivienda, agua, electricidad, gas y otros combustibles': 'ipc_vivienda',
    'Equipo y mantenimiento del hogar': 'ipc_hogar',
    'Salud': 'ipc_salud',
    'Transporte': 'ipc_transporte',
    'Comunicación': 'ipc_comunicacion',
    'Recreación y cultura': 'ipc_recreacion',
    'Educación': 'ipc_educacion',
    'Restaurantes y hoteles': 'ipc_restaurantes',
    'Bienes y servicios varios': 'ipc_otros',
})

# Filtrar IPC a los meses presentes en nuestra serie SEPA
ipc = ipc_raw[['mes','ipc_general','ipc_alimentos','ipc_bebidas_alc',
               'ipc_vestir','ipc_vivienda','ipc_hogar','ipc_salud',
               'ipc_transporte','ipc_comunicacion','ipc_recreacion',
               'ipc_educacion','ipc_restaurantes','ipc_otros']].copy()

# Convertir a numérico (puede haber comas como separador decimal)
for c in ipc.columns:
    if c != 'mes':
        ipc[c] = pd.to_numeric(
            ipc[c].astype(str).str.replace(',','.', regex=False),
            errors='coerce'
        )

# Calcular variaciones mensuales del IPC general y alimentos
ipc = ipc.sort_values('mes').reset_index(drop=True)
ipc['ipc_general_var_%'] = (ipc['ipc_general'].pct_change() * 100).round(2)
ipc['ipc_alimentos_var_%'] = (ipc['ipc_alimentos'].pct_change() * 100).round(2)

print(f"✅ IPC procesado: {len(ipc)} meses, desde {ipc['mes'].min()} hasta {ipc['mes'].max()}")
print(f"\nMuestra:")
print(ipc[['mes','ipc_general','ipc_general_var_%','ipc_alimentos','ipc_alimentos_var_%']].tail(10).to_string(index=False))

IPC INDEC cargado: 121 filas
Columnas: ['date', 'Nivel general', 'Alimentos y bebidas no alcohólicas', 'Bebidas Alcohólicas y tabaco', 'Prendas de vestir y calzado', 'Vivienda, agua, electricidad, gas y otros combustibles', 'Equipo y mantenimiento del hogar', 'Salud', 'Transporte', 'Comunicación', 'Recreación y cultura', 'Educación', 'Restaurantes y hoteles', 'Bienes y servicios varios']

✅ IPC procesado: 0 meses, desde nan hasta nan

Muestra:
Empty DataFrame
Columns: [mes, ipc_general, ipc_general_var_%, ipc_alimentos, ipc_alimentos_var_%]
Index: []


In [6]:
# ============================================================
# CELDA 6 — Construir tabla comparativa SEPA vs IPC
# ============================================================
# Merge de serie nacional válida con IPC
comparativa = serie_nacional_valida.merge(
    ipc[['mes','ipc_general','ipc_general_var_%','ipc_alimentos','ipc_alimentos_var_%']],
    on='mes', how='left'
)

# Reindexar el IPC general con base en el primer mes válido (para comparar índices)
mes_base = comparativa['mes'].iloc[0]
ipc_base = comparativa['ipc_general'].iloc[0]
comparativa['indice_ipc_general_base100'] = (comparativa['ipc_general'] / ipc_base * 100).round(2)

ipc_alim_base = comparativa['ipc_alimentos'].iloc[0]
comparativa['indice_ipc_alimentos_base100'] = (comparativa['ipc_alimentos'] / ipc_alim_base * 100).round(2)

# Diferencia entre la inflación de canasta y la de IPC (mensual y acumulada)
comparativa['brecha_var_vs_ipc_general_pp'] = (
    comparativa['variacion_mensual_%'] - comparativa['ipc_general_var_%']
).round(2)
comparativa['brecha_var_vs_ipc_alimentos_pp'] = (
    comparativa['variacion_mensual_%'] - comparativa['ipc_alimentos_var_%']
).round(2)

# Acumulado vs base
comparativa['brecha_acumulada_vs_ipc_general_pp'] = (
    comparativa['indice_canasta_base100'] - comparativa['indice_ipc_general_base100']
).round(2)
comparativa['brecha_acumulada_vs_ipc_alimentos_pp'] = (
    comparativa['indice_canasta_base100'] - comparativa['indice_ipc_alimentos_base100']
).round(2)

cols_resumen = ['mes','canasta_nacional_ponderada','variacion_mensual_%',
                'ipc_general_var_%','ipc_alimentos_var_%',
                'brecha_var_vs_ipc_general_pp','brecha_var_vs_ipc_alimentos_pp',
                'indice_canasta_base100','indice_ipc_general_base100','indice_ipc_alimentos_base100']
print("=== COMPARATIVA SEPA vs IPC (resumen) ===")
print(comparativa[cols_resumen].to_string(index=False))

=== COMPARATIVA SEPA vs IPC (resumen) ===
    mes  canasta_nacional_ponderada  variacion_mensual_%  ipc_general_var_%  ipc_alimentos_var_%  brecha_var_vs_ipc_general_pp  brecha_var_vs_ipc_alimentos_pp  indice_canasta_base100  indice_ipc_general_base100  indice_ipc_alimentos_base100
2022-01                     5305.37                  NaN                NaN                  NaN                           NaN                             NaN                  100.00                         NaN                           NaN
2022-02                     5551.10                 4.63                NaN                  NaN                           NaN                             NaN                  104.63                         NaN                           NaN
2022-03                     6003.10                 8.14                NaN                  NaN                           NaN                             NaN                  113.15                         NaN                         

In [7]:
# ============================================================
# CELDA 7 — Exportar a Excel consolidado y descargar
# ============================================================
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as w:
    # Hoja 1: serie nacional consolidada (válida)
    serie_nacional_valida.to_excel(w, sheet_name='nacional_valida', index=False)

    # Hoja 2: serie nacional completa (incluye descartados para trazabilidad)
    serie_nacional.to_excel(w, sheet_name='nacional_completa', index=False)

    # Hoja 3: serie por provincia (válida)
    serie_provincia_valida.to_excel(w, sheet_name='por_provincia', index=False)

    # Hoja 4: serie por región (válida)
    serie_region_valida.to_excel(w, sheet_name='por_region', index=False)

    # Hoja 5: IPC INDEC procesado
    ipc.to_excel(w, sheet_name='ipc_indec', index=False)

    # Hoja 6: comparativa SEPA vs IPC
    comparativa.to_excel(w, sheet_name='comparativa_sepa_ipc', index=False)

    # Hoja 7: documentación / notas
    notas = pd.DataFrame({
        'campo': [
            'Fuente datos SEPA',
            'Fuente IPC',
            'Productos en canasta',
            'Provincias',
            'Regiones',
            'Meses descartados',
            'Razón descarte',
            'Mes base índices',
            'Población ponderación',
            'Período cubierto',
        ],
        'valor': [
            'Ministerio de Economía - Sistema Electrónico de Publicidad de Precios Argentinos (SEPA)',
            'INDEC - Índice de Precios al Consumidor (IPC)',
            '30 productos de consumo masivo (Lácteos, Almacén, Bebidas, Limpieza, Higiene, Snacks)',
            f'{serie_provincia_valida["provincia"].nunique()} jurisdicciones',
            f'{serie_region_valida["region"].nunique()} regiones (AMBA, Pampeana, NOA, NEA, Cuyo, Patagonia)',
            ', '.join(MESES_DESCARTAR),
            'Cobertura SEPA insuficiente: productos como leche y fideos con < 30 observaciones/mes y reportados por 1 sola cadena',
            f'{serie_nacional_valida["mes"].iloc[0]} = 100',
            'Censo INDEC 2022 (45.892.285 habitantes en las 24 jurisdicciones)',
            f'{serie_nacional_valida["mes"].iloc[0]} a {serie_nacional_valida["mes"].iloc[-1]}',
        ]
    })
    notas.to_excel(w, sheet_name='metodologia', index=False)

print(f"✅ Archivo consolidado: {OUTPUT_FILE}")
print(f"   Tamaño: {os.path.getsize(OUTPUT_FILE)/1024:.0f} KB")

from google.colab import files
files.download(OUTPUT_FILE)

✅ Archivo consolidado: /content/canasta_SEPA_consolidado.xlsx
   Tamaño: 63 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>